# Workflow: Visium HD (segmented) {#sec-seq-workflow-visium-hd-seg}



## Preamble

### Introduction

As of June 2025, Visium HD enables direct output of H&E-segmented cell-level data, in addition to the previous `binned_outputs` at 2, 8, and 16 µm, from [SpaceRanger v4](https://www.10xgenomics.com/analysis-guides/segmentation-visium-hd). This update simplifies the additional cell segmentation required when using `bin2cell` or `StarDist`. However, a comparative study of segmentation methods on Visium HD data has yet to be conducted. 

The morphology-driven, nucleus-based segmentation produces polygons representing nuclei and cells across the entire tissue, as illustrated in the schematics provided by 10x Genomics.

In [ ]:
knitr::include_graphics("../images/VisiumHD_nuclei_segmentation_binning_10x.png")

### Dependencies

In [ ]:
library(sf)
library(arrow)
library(dplyr)
library(tidyr)
library(scrapper)
library(SingleR)
library(Voyager)
library(ggplot2)
library(ggspavis)
library(VisiumIO)
library(OSTA.data)
library(patchwork)
library(SpotSweeper)
library(DropletUtils)
library(SpatialExperiment)
library(SpatialFeatureExperiment)
# set seed for random number generation
# in order to make results reproducible
set.seed(777)

In [ ]:
# utility for cropping by bounding box
.crop <- \(spe, box) {
    box <- as.list(box)
    xy <- spatialCoords(spe)
    spe[, 
        xy[,1] > box$xmin & xy[,1] < box$xmax &
        xy[,2] > box$ymin & xy[,2] < box$ymax ]
}

## Setup

### From SPE...

Analogous to other chapters, we start out by retrieving data files from the OSF repository using `r BiocStyle::Biocpkg("OSTA.data")`. Output files from SpaceRanger v4 are located under `/segmented_outputs` (see @sec-bkg-importing-data for details on Visium HD output structure).
We can read these cell-level data into R as a `r BiocStyle::Biocpkg("SpatialExperiment")` [@Righelli2022-SpatialExperiment] using `r BiocStyle::Biocpkg("VisiumIO")`'s `TENxVisiumHD()` function by specifying the `segmented_outputs` argument accordingly. 

In [ ]:
# retrieve data from OSF repo
id <- "VisiumHD_HumanColon_Oliveira"
pa <- OSTA.data_load(id)
dir.create(td <- tempfile())
unzip(pa, exdir=td)

# read into 'SpatialExperiment'
seg <- file.path(td, "segmented_outputs")
spe <- TENxVisiumHD(
    format="h5", 
    images="lowres",
    segmented_outputs=seg) |>
    import()

# make gene symbols unique
gs <- rowData(spe)$Symbol
rownames(spe) <- make.unique(gs)

# needed for 'ggspavis'
spe$in_tissue <- TRUE

`SpaceRanger` also provides information on the mapping between binned outputs 
(2, 8, 16um) and segmented cells. These are stored as cell metadata slot `map`:

In [ ]:
# view bin-to-cell mapping information
head(spe$map[[1]])

Quick inspection shows us the number of 2um bins that were segmented, 
either as part of nuclei or whole cells:

In [ ]:
# number of segmented 2um bins
summary(sapply(spe$map, nrow))
# fraction of 2um bins assigned to nucleus
summary(sapply(spe$map, \(.) mean(.$in_nucleus)))

For runtime reasons, we will from the tissue 
to about half of its original width and height.

In [ ]:
xy <- spatialCoords(spe)
xs <- range(xy[, 1])
ys <- range(xy[, 2])
dx <- diff(xs)/4
dy <- diff(ys)/4
box <- list(
    xmin=xs[1]+dx, xmax=xs[2]-dx,
    ymin=ys[1]+dy, ymax=ys[2]-dy)
sub <- .crop(spe, box)

In [ ]:
#| code-fold: true
# reverse y-coordinates of bounding box
.box <- box
.box$ymax <- -box$ymin
.box$ymin <- -box$ymax
aes <- list(col="red", fill=NA, linewidth=2)
plotCoords(
    spe[, sample(ncol(spe), 5e4)]) + 
    do.call(geom_rect, c(.box, aes)) +
plotCoords(
    sub[, sample(ncol(sub), 5e4)])

This retains about `r round(100*ncol(sub)/ncol(spe), 2)`\% of cells, and `r ncol(sub)` in total.

In [ ]:
round(100*ncol(sub)/ncol(spe), 2)
ncol(spe <- sub)

We also specify a much smaller region of interest (ROI) for visualization purposes.

In [ ]:
roi <- c(xmin=54e3, xmax=58e3, ymin=12e3, ymax=16e3)
spe$roi <- colnames(spe) %in% colnames(.crop(spe, roi))
plotCoords(spe, annotate="roi", point_size=0) + theme(legend.position="none")

### ...to SFE

Next, we will convert the above SPE into a `r BiocStyle::Biocpkg("SpatialFeatureExperiment")` [@Moses2023-Voyager] that allows us to also store cell segmentation masks as a `colGeometry`.

In [ ]:
# convert from SPE to SFE
sfe <- toSpatialFeatureExperiment(spe)
# add cell segmentation boundaries
seg <- metadata(spe)$cellseg
i <- colnames(spe)
j <- match(i, seg$cell_id)
seg <- seg[j, ]; rownames(seg) <- i
colGeometries(sfe) <- list(cellseg=seg)

## Exploratory

### Cell masks

We can visualize such geometries using `plot(st_geometry(x))` where `x` is the `colGeometry` of interest. As `r ncol(sfe)` cells are a lot to visualize this way, we here `crop()` the data to filter for cells that fall within a small bounding box:

In [ ]:
par(mar=c(0,0,0,0))
# specify bounding box
box <- c(
    xmin=56e3, xmax=58e3, 
    ymin=15e3, ymax=16e3)
# plot exemplary cell masks
geo <- colGeometry(crop(sfe, box))
plot(st_geometry(geo), col=rep(colors(), 2))

### Bin mapping

We can estimate cell areas based on the number of 2um bins covered by each mask, e.g., 10 bins (4um$^2$ each) would correspond to an area of 40um$^2$. Based on this, we can also approximate the area of each cell's nucleus using the `in_nucleus` flag provided in the bin-to-cell mapping information.

In [ ]:
# get cell areas = 4x number of 2um bins
sfe$um2 <- 4*sapply(sfe$map, nrow)
# get fraction of 2um bins that are nuclear
sfe$nuc <- sapply(sfe$map, \(.) mean(.$in_nucleus))
# get nucleus areas = cell area x nuclear fraction
sfe$nuc_um2 <- sfe$um2*sfe$nuc

We can also investigate the number of different cells
that are contained in 8um and 16um bins, respectively:

In [ ]:
names(bin) <- bin <- c("square_008um", "square_016um")
ncs <- lapply(bin, \(um) {
    ns <- lapply(spe$map, \(df) unique(df[[um]]))
    as.vector(table(unlist(ns)))
})
sapply(ncs, summary)

In [ ]:
max <- sapply(ncs, max)
med <- sapply(ncs, median)
q75 <- sapply(ncs, quantile, 0.75)

Here, we can observe that 8/16um bins map to a median of `r med[1]`/`r med[2]` 
cells, only 25\% of bins map to more than `r q75[1]`/`r q75[2]` cells, and at 
most `r max[1]`/`r max[2]` cells are mapped to any bin.

## Quality control

### Non-spatial

We can caluclate standard cell-level quality control metrics using 
`r BiocStyle::Biocpkg("scrapper")`'s `quickRnaQc.se()` function,
specifying mitochondrial features as `subsets` of particular interest.

In [ ]:
mt <- grepl("^MT-", rownames(sfe))
sfe <- quickRnaQc.se(sfe, subsets=list(mt=mt))
sfe$log_sum <- log1p(sfe$sum)
head(colData(sfe)[c(
    "um2", "nuc_um2", "nuc",
    "sum", "subset.proportion.mt")])

In [ ]:
#| code-fold: true
plotSpatialFeature(sfe[, sfe$roi], 
    colGeometryName="cellseg", 
    features="log_sum") +
    ggtitle("log-library size") +
plotSpatialFeature(sfe[, sfe$roi], 
    colGeometryName="cellseg", 
    features="subset.proportion.mt") +
    ggtitle("mitochondrial proportion") +
plot_layout(nrow=1) &
    theme(plot.title=element_text(hjust=0.5)) &
    scale_fill_gradientn(NULL, colors=pals::jet())

### Spatially-aware

`r BiocStyle::Biocpkg("Spotsweeper")` [@Totty2025-SpotSweeper] determines 
outliers based on low log-library size, few uniquely detected features, or 
a high mitochondrial fraction compared to their surrounding neighborhood, 
which we recommend (see @sec-seq-quality-control).

In [ ]:
# determine spatial outliers for different metrics
sfe <- localOutliers(sfe,
    workers=4, metric="sum",
    log=TRUE, direction="lower")
sfe <- localOutliers(sfe,
    workers=4, metric="detected",
    log=TRUE, direction="lower")
sfe <- localOutliers(sfe,
    workers=4, metric="subset.proportion.mt",
    log=FALSE, direction="higher")
# get cells x flags matrix
cd <- colData(sfe)
ol <- grep("outliers$", names(cd))
ol <- as.matrix(cd[ol])
# percentage of cells exluded for each 
round(100*colMeans(ol), 2)
# number of cells kept/removed overall
table(sfe$ex <- rowAnys(ol))

We can visualize local outliers identified by `SpotSweeper` spatially. 
However, these are difficult to see, as only small cells were flagged.

In [ ]:
#| code-fold: true
plotCoords(sfe[, order(sfe$ex)], 
    annotate="ex", point_size=0.2) +
    theme(legend.key.size=ggplot2::unit(0, "pt")) +
    scale_color_manual(values=c("grey90", "red")) +
    guides(col=guide_legend(override.aes=list(size=2)))

In [ ]:
# apply filtering
sfe <- sfe[, !sfe$ex]

## Annotation

This Visium HD dataset provides matched scRNA-seq for the same tissue, 
which avoids tissue and batch effects that often complicate label transfer. 
`colData` slot `Level1` contains broad, interpretable annotations that are 
well suited for label-transfer, deconvolution, and downstream interpretation. 
We will use these to map likely cell identities onto our Visium HD cells.

::: {.callout-note collapse="true" title="using deconvolution instead"}

We here use a single-cell label transfer approach to annotade cells. However, **although each observation is intended to represent a single cell, there are still cases where this is not the case** (e.g., as a result of partial volume effects at boundaries, segmentation errors, tightly packed regions, or local diffusion of transcripts.

Once could hence deconvolute cells instead (see @sec-seq-deconvolution), for example, using `r BiocStyle::Biocpkg("spacexr")`'s `RCTD` [@Cable2022-RCTD] **in doublet mode** to flag these cases and provide a principled estimate of the most likely composition. In practice, we expect most units to be classified as singlets, with a minority of doublets that warrant closer inspection or additional filtering.

:::

In [ ]:
# retrieve single-cell reference from OSF repo
id <- "Chromium_HumanColon_Oliveira"
pa <- OSTA.data_load(id)
dir.create(td <- tempfile())
unzip(pa, exdir=td)

# read into 'SingleCellExperiment'
fs <- list.files(td, full.names=TRUE)
h5 <- grep("h5$", fs, value=TRUE)
sce <- read10xCounts(h5, col.names=TRUE)

# add cell metadata
csv <- grep("csv$", fs, value=TRUE)
cd <- read.csv(csv, row.names=1)
colData(sce)[names(cd)] <- cd[colnames(sce), ]

# use gene symbols as feature names
rownames(sce) <- make.unique(rowData(sce)$Symbol)

# exclude cells deemed to be of low-quality
sce <- sce[, sce$QCFilter == "Keep"]

# replace problematic characters
table(sce$Level1 <- gsub("\\s", "\\.", sce$Level1))

For the purpose of this demo, we downsample to at most 1,000 cells per reference class. 
This preserves diversity within each class while keeping memory and runtime manageable.

In [ ]:
# (this is done only to keep runtime/memory low)
idx <- split(seq_len(ncol(sce)), sce$Level1)
idx <- lapply(idx, \(.) sample(., min(length(.), 1e3)))
ncol(.sce <- sce[, unlist(idx)])

We next run `BiocStyle::Biocpkg("SingleR")` using `Level1` (low-resolution) 
annotations and with argument `aggr.ref=TRUE`, such that reference profiles 
will be aggregated (per cluster) prior to annotation, and every cell will be
assigned the label of the best-fitting pseudo-bulk scRNA-seq reference profile.

In [ ]:
# log-library size normalization
sfe <- normalizeRnaCounts.se(sfe)
.sce <- normalizeRnaCounts.se(.sce)
# perform label transfer at the Visium HD cell-level,
# using pseudo-bulk Chromium profiles as reference
res <- SingleR(test=sfe, ref=.sce, labels=.sce$Level1, aggr.ref=TRUE)
sfe$Level1 <- factor(res$pruned.labels)

Simplifying further, we can group cells into different compartments, namely, 
(malignant) tumor, immune, epithelial and stromal cells; we’ll see below that 
visualizing cells in this way nicely captures the general tissue structure.

In [ ]:
lab <- list(
    tum=c("Tumor"),
    epi=c("Intestinal.Epithelial"),
    imm=c("B.cells", "T.cells", "Myeloid"),
    str=c("Endothelial", "Fibroblast", "Smooth.Muscle"))
idx <- match(sfe$Level1, unlist(lab))
lab <- rep.int(names(lab), sapply(lab, length))
table(sfe$Level0 <- factor(lab[idx]))

We can visualize low-resolution and compartment-level annotations over our ROI.

In [ ]:
#| code-fold: true
pal_lv1 <- unname(pals::trubetskoy())
pal_lv0 <- c("gold", "turquoise", "deeppink", "navy")
plotSpatialFeature(sfe[, sfe$roi], 
    features="Level1", colGeometryName="cellseg") + 
    scale_fill_manual(values=pal_lv1) +
plotSpatialFeature(sfe[, sfe$roi], 
    features="Level0", colGeometryName="cellseg") + 
    scale_fill_manual(values=pal_lv0) +
plot_layout(nrow=1) &
    theme(legend.key.size=ggplot2::unit(.5, "lines"))

For whole-section visualization, we do not render 
segmentation boundaries but only cell centroids. 

In [ ]:
#| code-fold: true
plotCoords(sfe, 
    annotate="Level1", point_size=0.1) +
    scale_color_manual(values=pal_lv1) +
plotCoords(sfe, 
    annotate="Level0", point_size=0.1) +
    scale_color_manual(values=pal_lv0) +
plot_layout() &
    theme(legend.key.size=ggplot2::unit(0, "pt"))

## Downstream

Having annoted our cells, we can proceed with various downstream analyses.
Compared to analysis at the bin-level, cell-level Visium HD data is similar
to data from imaging-based platforms, but with full-transcriptome coverage.
In princple, most analyses are thus transferable. 

We refer readers to the chapters designed to specific analysis tasks, e.g.,
@sec-ind-feature-selection-testing on feature selection and testing, 
and @sec-ind-feature-set-signatures on feature-set scoring.

Instead, we quickly inspect how cell and nucleus areas differ across 
compartments and subpopulations. As expected, immune cells are smallest, 
tumor and structural cells (epithelial and stromal cells) tend to be larger.

In [ ]:
#| code-fold: true
# wrangling
df <- data.frame(colData(sfe))
df <- filter(df, !is.na(Level0))
fd <- pivot_longer(df, ends_with("um2"))
fd$name <- factor(fd$name, labels=c("nuc.", "cell"))
ws <- c(nlevels(df$Level0), nlevels(df$Level1))
# plotting
ggplot(fd, aes(reorder(Level0, value), value, col=name)) +
ggplot(fd, aes(reorder(Level1, value), value, col=name)) + 
plot_layout(nrow=1, widths=ws, guides="collect") &
    geom_boxplot(key_glyph="point", outlier.size=0) &
    labs(col="area (um2)") & 
    theme_bw() & theme(
        axis.title=element_blank(),
        legend.key.size=ggplot2::unit(0, "pt"),
        axis.text.x=element_text(angle=45, hjust=1))  

## Appendix

This workflow is partly based on a workshop that has been taught at the European Bioconductor Conference 2025 (EuroBioC2025). Corresponding workshop material can be found [here](https://estellad.github.io/EuroBioC2025OSTAWorkshop/index.html).

### References {.unnumbered}